### **广义加性模型**

#### **一、基本介绍**

广义加性模型（Generalized Additive Models，GAM）是一种统计模型，用于描述响应变量与多个预测变量之间的关系。

设待拟合样本集为：


$$
\{(x_{k,1}, x_{k,2}, \cdots, x_{k,p}, y_k)\mid 1 \leq k \leq N\}
$$

其中，$k$ 为样本索引，总样本量为 $N$，特征维数为 $p$，散点图如下所示：

<div align="center">
<img src="./img/图1.png" alt="图1" width="300">
</div>


而GAMs的数学式如下：

$$
y_k = \beta_0 + f_1(x_{k,1}) + f_2(x_{k,2}) + \cdots + f_p(x_{k,p}) + \varepsilon_k
$$

其中，$y_k$ 是第 $k$ 个样本的响应变量，$\beta_0$ 是截距项，$f_i(\cdot)\ (1\le i\le p)$ 是第 $i$ 个预测变量的非线性平滑函数；在高斯回归设定下，$\varepsilon_k$ 可假设为零均值正态噪声。

#### **二、单特征拟合**

如果只有一个预测变量，则GAMs的数学式可以简化为：

$$
y_k = \beta_0 + f_1(x_{k,1}) + \varepsilon_k
$$

略去特征下标，有：

$$
y_k = \beta_0 + f_1(x_k) + \varepsilon_k
$$

如下图所示：

<div align="center">
<img src="./img/图2.png" alt="图1" width="300">
</div>

接下来，讨论平滑函数 $f_1$。它可以是任何形式的函数，例如多项式函数、样条函数、核函数等。常见的平滑函数包括：
- 样条函数：$f_1(x) = \sum_{j=1}^m \beta_j B_j(x)$，其中 $B_j(x)$ 是第 $j$ 个样条基函数，$m$ 是样条基函数的数量。
- 多项式函数：$f_1(x) = \beta_1 x + \beta_2 x^2 + \cdots + \beta_d x^d$，其中 $d$ 是多项式的阶数。
- 平滑样条：$f_1(x) = \sum_{j=1}^m \beta_j B_j(x)$，通过惩罚项控制函数的平滑程度，常用于拟合复杂的非线性关系。
- 核函数回归：$f_1(x) = \sum_{j=1}^N \alpha_j K(x, x_j)$，其中 $K(\cdot, \cdot)$ 是核函数，$\alpha_j$ 是核函数的权重。
- 局部加权回归：$f_1(x) = \sum_{j=1}^N w_j(x) y_j$，其中 $w_j(x)$ 是第 $j$ 个样本对 $x$ 的权重，常用于拟合局部非线性结构。

#### **2.1 单特征单点拟合**

这里以样条函数为例，说明如何拟合平滑函数 $f_1$。假设我们选择了 $m$ 个基函数 $B_j(x)$ 来拟合单个样本点 $(x_k, y_k)$，则可以将 $f_1(x_k)$ 表示为：

$$
f_1(x_k) = \sum_{j=1}^m \beta_j B_j(x_k)
$$

这意味着，响应变量 $y_k$ 可以表示为 $m$ 个基函数的线性组合：

$$
y_k = \beta_0 + \sum_{j=1}^m \beta_j B_j(x_k) + \varepsilon_k
$$

在实际应用中，一般选择B样条函数作为平滑函数的基函数，因为它具有良好的拟合能力和数值稳定性。通过调整基函数的数量 $m$ 和位置，可以控制模型的复杂度和拟合效果[1](https://pages.mtu.edu/~shene/COURSES/cs3621/NOTES/spline/B-spline/bspline-basis.html)。下图展示了采用三角形基函数，且基函数个数设置为 $m=3$ 时对样本 $(x_k, y_k)$ 的拟合原理：

<div align="center">
<img src="./img/图3.png" alt="图1" width="300">
</div>

#### **2.2 单特征多点拟合**

更进一步地，考虑采用 $m$ 个基函数同时拟合 $N$ 个样本点 $(x_k, y_k)$，则可以将所有样本点的响应变量表示为：

$$
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{bmatrix}
=
\begin{bmatrix}
1 & B_1(x_1) & B_2(x_1) & \cdots & B_m(x_1) \\
1 & B_1(x_2) & B_2(x_2) & \cdots & B_m(x_2) \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & B_1(x_N) & B_2(x_N) & \cdots & B_m(x_N)
\end{bmatrix}
\begin{bmatrix}
\beta_0 \\
\beta_1 \\
\beta_2 \\
\vdots \\
\beta_m
\end{bmatrix}
+
\begin{bmatrix}
\varepsilon_1 \\
\varepsilon_2 \\
\vdots \\
\varepsilon_N
\end{bmatrix}
$$

其中，第一列全为1，对应截距项 $\beta_0$；后续列对应各个基函数 $B_j(x_k)$ 的值。通过最小二乘法或其他优化方法，可以求解出参数 $\beta_j$ 的估计值，从而得到平滑函数 $f_1(x)$ 的拟合结果。可通过梯度下降法等数值优化算法来求解参数 $\beta_j$，以最小化残差平方和：

$$
\min_{\beta_0, \beta_1, \cdots, \beta_m} \sum_{k=1}^N \left( y_k - \beta_0 - \sum_{j=1}^m \beta_j B_j(x_k) \right)^2
$$

#### **2.3 惩罚回归**

若仅通过最小化残差平方和来求解参数，当基函数个数 $m$ 较大时，模型虽然能够更好地贴合训练数据，但也更容易出现过拟合，即拟合曲线过于波动，泛化能力下降。

为了解决这一问题，可以在原有损失函数中加入一个**惩罚项**，用于约束平滑函数的复杂度。于是，优化目标可写为：

$$
\min_{\beta_0,\beta_1,\cdots,\beta_m}
\sum_{k=1}^{N}\left(y_k-\beta_0-\sum_{j=1}^{m}\beta_j B_j(x_k)\right)^2
+
\lambda J(f_1)
$$

其中：

- 第一项是残差平方和，用于衡量模型对数据的拟合误差；
- 第二项 $\lambda J(f_1)$ 是惩罚项，用于控制函数 $f_1$ 的平滑程度；
- $\lambda \ge 0$ 为平滑参数，用于平衡“拟合能力”与“平滑程度”。

通常，惩罚项 $J(f_1)$ 可以取为函数曲率的平方积分，例如：

$$
J(f_1)=\int \left(f_1''(x)\right)^2 dx
$$

该形式表示：若函数二阶导数较大，说明曲线弯折较剧烈，则惩罚值更大；反之，若函数较平滑，则惩罚值较小。因此，惩罚回归的本质是在“尽量拟合数据”和“避免函数过于复杂”之间取得折中。

平滑参数 $\lambda$ 的作用可概括为：

- 当 $\lambda=0$ 时，不施加任何平滑约束，模型退化为普通最小二乘拟合；
- 当 $\lambda$ 较小时，模型更关注拟合训练数据；
- 当 $\lambda$ 较大时，模型更关注函数的平滑性；
- 当 $\lambda \to \infty$ 时，函数会被强烈约束，趋向于非常简单的形式。

若将样本写成矩阵形式，则惩罚回归可进一步表示为：

$$
\min_{\boldsymbol{\beta}}
\|\mathbf{y}-\mathbf{B}\boldsymbol{\beta}\|^2
+
\lambda \boldsymbol{\beta}^\top \mathbf{\Omega}\boldsymbol{\beta}
$$

其中：

- $\mathbf{y}$ 是响应变量向量；
- $\mathbf{B}$ 是由基函数构成的设计矩阵；
- $\boldsymbol{\beta}$ 是待估参数向量；
- $\mathbf{\Omega}$ 是由基函数及其导数构造出的惩罚矩阵。

这样，原问题就转化为了一个带正则化项的最优化问题。对GAM而言，惩罚回归是核心思想之一，因为它既保留了非线性拟合能力，又能有效抑制过拟合，使得到的平滑函数更稳定、更具有解释性。